<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [25]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from config.spark_config import SparkConfig
from config.io_config import *
from app.platform_app import PlatformApp
from transformation.gold_loader import *
from utils.logger import LoggerConfig
from transformation.bronze_ingestion import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *

In [26]:
logger = LoggerConfig().setup_logger(log_dir=None)
spark = SparkConfig.create_spark(app_name="FMCG Domain", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="fmcg_domain")

2026-03-22 21:30:53 | INFO     | Connected to Databricks via Spark Connect.
2026-03-22 21:30:53 | INFO     | Initializing Data Platform...
2026-03-22 21:30:53 | INFO     | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Bronze</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Bronze
    </h1>
</div>


In [27]:
# Bronze Ingestion
logger.info("Ingesting data into Bronze layer...")
upload_file_to_bronze(spark=spark, logger=logger, entity="orders",
                      path_data=S3_BASE_PATH, path_cp=CP_PATH_BRONZE,
                      name_catalog=app.catalog_name)
logger.info("Bronze ingestion completed.")
logger.info("*" * 80)

2026-03-22 21:30:53 | INFO     | Ingesting data into Bronze layer...
2026-03-22 21:30:53 | INFO     | Starting Bronze ingestion for entity: Orders
2026-03-22 21:30:53 | INFO     | Uploading file: s3://00-sportsbar-dp/orders/landing
2026-03-22 21:31:09 | INFO     | Orders: Schema inferred successfully
2026-03-22 21:31:21 | INFO     | Completed Bronze ingestion for entity: Orders
+------------+---------------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|order_id    |order_placement_date       |customer_id|product_id|order_qty|read_timestamp        |file_name            |file_size|file_modification_time|
+------------+---------------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|FOCT62720602|Tuesday, September 30, 2025|ABC987     |25891301  |71.0     |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FO

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [28]:
df_silver = spark.sql(f"SELECT * FROM {app.catalog_name}.bronze.orders")
df_silver.show(truncate=False)

+------------+---------------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|order_id    |order_placement_date       |customer_id|product_id|order_qty|read_timestamp        |file_name            |file_size|file_modification_time|
+------------+---------------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|FOCT62720602|Tuesday, September 30, 2025|ABC987     |25891301  |71.0     |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|Tuesday, September 30, 2025|789720     |25891502  |125.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|Tuesday, September 30, 2025|789720     |25891403  |462.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|Tuesday, September 30, 2025|INVALID    |25891601  |133.0    |2

## Transformations

### Check NULL

In [29]:
# check null
check_null(df = df_silver, logger=logger)

2026-03-22 21:31:26 | WARNING  | 
+-----------+---------------+-----------+
| Features  | Missing_Count | Missing_% |
+-----------+---------------+-----------+
| order_qty |     2607      |   5.03    |
+-----------+---------------+-----------+
2026-03-22 21:31:26 | WARNING  | Total missing values: 2,607 out of 51,810 rows.


### Trim spaces in customer name

In [30]:
# remove those trim values
df_silver = clean_dataframe(df=df_silver)
df_silver.show(truncate=False)

+------------+---------------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|order_id    |order_placement_date       |customer_id|product_id|order_qty|read_timestamp        |file_name            |file_size|file_modification_time|
+------------+---------------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|FOCT62720602|Tuesday, September 30, 2025|ABC987     |25891301  |71.0     |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|Tuesday, September 30, 2025|789720     |25891502  |125.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|Tuesday, September 30, 2025|789720     |25891403  |462.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|Tuesday, September 30, 2025|INVALID    |25891601  |133.0    |2

### Transformations

In [31]:
df_silver.select("order_placement_date").distinct().show(truncate=False)

+-----------------------------+
|order_placement_date         |
+-----------------------------+
|Tuesday, September 30, 2025  |
|2025/09/30                   |
|30-09-2025                   |
|30/09/2025                   |
|Sunday, November 30, 2025    |
|30-11-2025                   |
|2025/11/30                   |
|30/11/2025                   |
|13-10-2025                   |
|Monday, October 13, 2025     |
|13/10/2025                   |
|2025/10/13                   |
|17-09-2025                   |
|2025/09/17                   |
|Wednesday, September 17, 2025|
|17/09/2025                   |
|Thursday, September 11, 2025 |
|11/09/2025                   |
|11-09-2025                   |
|2025/09/11                   |
+-----------------------------+
only showing top 20 rows


In [32]:
# Keep only rows where order_qty is present
df_silver = df_silver.filter(F.col("order_qty").isNotNull())

# Clean customer_id → keep numeric, else set to 999999
df_silver = (
    df_silver
    .withColumn(
        "customer_id",
        F.when(
            F.col("customer_id").cast("string").rlike(r"^[0-9]+$"), F.col("customer_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
)

# Remove weekday name from the date text
# "Tuesday, July 01, 2025" → "July 01, 2025"
df_silver = (
    df_silver
    .withColumn(
        "order_placement_date",
        F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
    )
)

# Parse order_placement_date using multiple possible formats
# coalesce(col1, col2, col3)
# nếu col1 != NULL → dùng col1
# nếu col1 = NULL → dùng col2
# nếu col2 = NULL → dùng col3
df_silver = df_silver.withColumn(
    "order_placement_date",
    F.coalesce(
        F.expr("try_cast(order_placement_date as date)"),
        F.expr("try_to_date(order_placement_date,'yyyy/MM/dd')"),
        F.expr("try_to_date(order_placement_date,'dd-MM-yyyy')"),
        F.expr("try_to_date(order_placement_date,'dd/MM/yyyy')"),
        F.expr("try_to_date(order_placement_date,'MMMM dd, yyyy')")
    )
)

# convert product id to string
df_silver = df_silver.withColumn("product_id", F.col("product_id").cast("string"))

df_silver.show(truncate=False)

+------------+--------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|order_id    |order_placement_date|customer_id|product_id|order_qty|read_timestamp        |file_name            |file_size|file_modification_time|
+------------+--------------------+-----------+----------+---------+----------------------+---------------------+---------+----------------------+
|FOCT62720602|2025-09-30          |999999     |25891301  |71.0     |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|2025-09-30          |789720     |25891502  |125.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|2025-09-30          |789720     |25891403  |462.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446    |2026-03-05 13:07:09   |
|FOCT62720602|2025-09-30          |999999     |25891601  |133.0    |2026-03-21 05:37:22.18|orders_2025_09_30.csv|41446

### Duplicates

In [33]:
check_duplicate(df=df_silver, logger=logger)

2026-03-22 21:31:32 | WARNING  | 3,616 duplicate rows detected (7.35%).
2026-03-22 21:31:32 | WARNING  | Rows affected: 3,616 out of 49,203.


In [34]:
# # Drop duplicates
# columns = ["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"]
# df_silver = df_silver.dropDuplicates(subset=columns)
# df_silver.count()

In [35]:
columns = ["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"]
df_silver = dedup(df=df_silver, dedup_cols=columns, 
                  cdc="read_timestamp", logger=logger)

2026-03-22 21:31:32 | INFO     | Starting deduplication
2026-03-22 21:31:32 | INFO     | Dedup columns: ['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'] | CDC column: read_timestamp
2026-03-22 21:31:32 | INFO     | Input row count: 49203
2026-03-22 21:31:35 | INFO     | Output row count after dedup: 45587
2026-03-22 21:31:35 | INFO     | Removed duplicate rows: 3616
2026-03-22 21:31:35 | INFO     | Deduplication completed


### check what's the maximum and minimum date

In [36]:
df_silver.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show(truncate=False)

+----------+----------+
|min_date  |max_date  |
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



### Join with products

In [37]:
df_products = spark.table(f"{app.catalog_name}.silver.dim_products")
df_joined = (
    df_silver
    .join(df_products, on="product_id", how="inner")
    .select(df_silver["*"], df_products["product_code"])
)
df_joined = df_joined.select("order_id", "order_placement_date", "customer_id", "product_id", 
                             "product_code", "order_qty", "read_timestamp", "file_name", "file_size", "file_modification_time")
df_joined.show(truncate=False)

+-------------+--------------------+-----------+----------+----------------------------------------------------------------+---------+----------------------+---------------------+---------+----------------------+
|order_id     |order_placement_date|customer_id|product_id|product_code                                                    |order_qty|read_timestamp        |file_name            |file_size|file_modification_time|
+-------------+--------------------+-----------+----------+----------------------------------------------------------------+---------+----------------------+---------------------+---------+----------------------+
|FAUG410101302|2025-08-08          |789101     |25891103  |102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|493.0    |2026-03-21 05:37:22.18|orders_2025_08_08.csv|21432    |2026-03-05 13:07:37   |
|FAUG410101302|2025-08-08          |789101     |25891203  |889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2|374.0    |2026-03-21 05:

### Transformed data to Silver Layer

In [38]:
key_cols = ["order_id", "order_placement_date", "customer_id", "product_id", "product_code"]
if not spark.catalog.tableExists(SILVER_ORDERS):
    logger.info("Silver Orders table not found. Creating new table...")
    df_joined.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("overwrite") \
                   .saveAsTable(SILVER_ORDERS)
    logger.info("Silver Orders table created successfully")
else:
    logger.info("Silver Orders table exists. Performing upsert...")
    upsert(spark=spark, df=df_joined, key_cols=key_cols,
           table="fact_orders", cdc="read_timestamp",
           name_catalog=app.catalog_name, name_schema="silver", logger=logger)
    logger.info("Upsert completed successfully")

2026-03-22 21:31:39 | INFO     | Silver Orders table exists. Performing upsert...
2026-03-22 21:31:39 | INFO     | Starting UPSERT into fmcg_domain.silver.fact_orders
2026-03-22 21:31:48 | INFO     | UPSERT completed successfully: fmcg_domain.silver.fact_orders
2026-03-22 21:31:48 | INFO     | Upsert completed successfully


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Gold</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Gold
    </h1>
</div>


In [39]:
df_gold = spark.sql(f"""
    SELECT  order_id, 
            order_placement_date as date, 
            customer_id as customer_code, 
            product_code,
            product_id, 
            order_qty as sold_quantity 
            FROM {SILVER_ORDERS};
""")
df_gold.show(truncate=False)

+------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
|order_id    |date      |customer_code|product_code                                                    |product_id|sold_quantity|
+------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
|FSEP59903603|2025-09-06|789903       |451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|25891603  |91.0         |
|FSEP59903601|2025-09-08|789903       |716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|25891601  |86.0         |
|FSEP59903503|2025-09-08|789903       |062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|25891503  |121.0        |
|FSEP59903503|2025-09-08|789903       |e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|25891102  |294.0        |
|FSEP59903502|2025-09-07|789903       |ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387

In [40]:
logger.info("Gold Gross Price table not found. Creating new table...")
df_gold.write.format("delta") \
                .option("delta.enableChangeDataFeed", "true") \
                .option("mergeSchema", "true") \
                .mode("overwrite") \
                .saveAsTable(GOLD_SB_ORDERS)
logger.info("Gold Gross Price table created successfully")

2026-03-22 21:31:50 | INFO     | Gold Gross Price table not found. Creating new table...
2026-03-22 21:31:52 | INFO     | Gold Gross Price table created successfully


## Merging with Parent company

In [41]:
df_gold_parent = spark.table(f"{app.catalog_name}.gold.fact_orders")
df_gold_parent.show(truncate=False)

+----------+------------+-------------+-------------+
|date      |product_code|customer_code|sold_quantity|
+----------+------------+-------------+-------------+
|2024-01-01|ARCHDDE20D  |70002017     |161          |
|2024-01-01|ARCH158F41  |70002017     |133          |
|2024-01-01|ARCHAFF0E4  |70002017     |76           |
|2024-01-01|ARCH6B94F7  |70002017     |92           |
|2024-01-01|ARCH5D1FE7  |70002017     |117          |
|2024-01-01|ARCH7B49A9  |70002017     |36           |
|2024-01-01|ARCH497D34  |70002017     |98           |
|2024-01-01|ARCHE71D79  |70002017     |156          |
|2024-01-01|BADM88C384  |70002017     |28           |
|2024-01-01|BADMA5EBA3  |70002017     |33           |
|2024-01-01|BADM4BAB38  |70002017     |22           |
|2024-01-01|BADMDC6AB1  |70002017     |26           |
|2024-01-01|BADM50C510  |70002017     |15           |
|2024-01-01|BADM2459FB  |70002017     |21           |
|2024-01-01|BADM9DDCED  |70002017     |68           |
|2024-01-01|BADMD11DA6  |700

In [42]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {GOLD_SB_ORDERS}")
df_child.show(truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-09-06|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|789903       |91.0         |
|2025-09-08|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|789903       |86.0         |
|2025-09-08|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|789903       |121.0        |
|2025-09-08|e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|789903       |294.0        |
|2025-09-07|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789903       |106.0        |
|2025-09-07|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789903       |202.0        |
|2025-09-07|0cb7b2f42657b625f754e833aa1cf6a967

In [43]:
df_monthly = (
    df_child
    # Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.date_trunc("month", "date").cast("date"))
    # Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
    # Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-09-01|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|789903       |1696.0       |
|2025-09-01|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|789903       |1434.0       |
|2025-09-01|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|789903       |2555.0       |
|2025-09-01|e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|789903       |6166.0       |
|2025-09-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789903       |2064.0       |
|2025-09-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789903       |3286.0       |
|2025-09-01|0cb7b2f42657b625f754e833aa1cf6a967

In [44]:
gold_parent_delta = DeltaTable.forName(spark, f"{app.catalog_name}.gold.fact_orders")
(gold_parent_delta
 .alias("parent_gold")
 .merge(
     source=df_monthly.alias("child_gold"), 
     condition="parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code"
    )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute()
 )

,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,3060,0,0,3060
